In [ ]:
import json
from datetime import datetime

import altair as alt
import polars as pl

In [ ]:
alt.data_transformers.disable_max_rows()


def theme():
    with open("camminady_theme.json") as f:
        return json.load(f)


alt.themes.register("_my_theme", theme)
alt.themes.enable("_my_theme")

In [ ]:
df = pl.read_csv(
    "https://raw.githubusercontent.com/thomascamminady/traffic-balve/main/data/summary.csv"
)

In [ ]:
(
    alt.Chart(
        df.with_columns(pl.col("datetime").cast(pl.Datetime))  # type: ignore
        .with_columns(duration_at_50kph_s=pl.col("distance_m") / 1000 / 50 * 3600)
        .with_columns(
            delay_m=(pl.col("duration_in_traffic_s") - pl.col("duration_at_50kph_s"))
            / 60
        )
        .with_columns(pl.col("from_to").str.replace("->", "→"))
        .sort("datetime", descending=False)
        .rolling(
            pl.col("datetime"),
            period=f"{(period:=60)}m",
            offset=f"-{period//2}m",
            by=["from", "to"],
        )
        .agg(pl.col("delay_m").mean(), from_to=pl.col("from_to").first())
        .with_columns(
            today=pl.col("datetime").dt.date() == datetime.now().date(),
        )
        .with_columns(
            cat=pl.col("from_to").replace(
                {
                    "Höhle → Krankenhaus": "Höhle ↔ Krankenhaus",
                    "Krankenhaus → Höhle": "Höhle ↔ Krankenhaus",
                    "Höhle → Krumpaul": "Krumpaul ↔ Höhle",
                    "Krumpaul → Höhle": "Krumpaul ↔ Höhle",
                    "Krumpaul → Krankenhaus": "Krankenhaus ↔ Krumpaul",
                    "Krankenhaus → Krumpaul": "Krankenhaus ↔ Krumpaul",
                }
            )
        )
    )
    # .mark_point(filled=True)
    .mark_line(point=False, clip=True)
    .encode(
        x=alt.X("hoursminutes(datetime):T").title("Uhrzeit"),
        y=alt.Y("delay_m:Q").scale(zero=False, domainMin=0).title("Verspätung [min]"),
        detail="date(datetime):T",
        row=alt.Row("from:N", spacing=100).title("Von"),
        color=alt.condition(  # type: ignore
            alt.datum.today,
            alt.Color("cat:N").title("Verbindung", labelLimit=0),
            alt.value("gray"),
        ),
        # color=alt.Color("cat:N").title("Verbindung"),
        column=alt.Column("to:N").title("Nach"),
        opacity=alt.condition(  # type: ignore
            alt.datum.today,
            alt.value(1.0),
            alt.value(0.2),
        ),
        strokeWidth=alt.condition(  # type: ignore
            alt.datum.today,
            alt.value(5),
            alt.value(2),
        ),
        tooltip=["datetime:T", "delay_m:Q"],
    )
    .configure_header(  # type: ignore
        labelColor="gray",
        labelFontSize=15,
        labelAngle=0,
        labelAlign="left",
        titleColor="gray",
        titleFontSize=20,
    )
    .properties(width=500, height=200)
)

In [ ]:
(
    alt.Chart(
        df.with_columns(pl.col("datetime").cast(pl.Datetime))  # type: ignore
        .with_columns(duration_at_50kph_s=pl.col("distance_m") / 1000 / 50 * 3600)
        .with_columns(
            delay_m=(pl.col("duration_in_traffic_s") - pl.col("duration_at_50kph_s"))
            / 60
        )
        .with_columns(pl.col("from_to").str.replace("->", "→"))
        .sort("datetime", descending=False)
        .rolling(
            pl.col("datetime"),
            period=f"{(period:=15)}m",
            offset=f"-{period//2}m",
            by=["from", "to"],
        )
        .agg(pl.col("delay_m").mean(), from_to=pl.col("from_to").first())
        .with_columns(
            today=pl.col("datetime").dt.date() == datetime.now().date(),
        )
    )
    .mark_line(point=False, clip=True)
    .encode(
        x=alt.X("hoursminutes(datetime):T").title("Uhrzeit"),
        y=alt.Y("delay_m:Q").scale(zero=False, domainMin=0).title("Verspätung [min]"),
        detail="date(datetime):T",
        color=alt.Color("to:N").title("Nach", labelLimit=0),
        opacity=alt.condition(  # type: ignore
            alt.datum.today,
            alt.value(1.0),
            alt.value(0.1),
        ),
        strokeWidth=alt.condition(  # type: ignore
            alt.datum.today,
            alt.value(4),
            alt.value(2),
        ),
        tooltip=["from_to:N", "datetime:T", "delay_m:Q"],
        row=alt.Row("from:N", spacing=50).title(None),
    )
    .configure_header(  # type: ignore
        labelColor="gray",
        labelFontSize=15,
        labelAngle=0,
        labelAlign="left",
        titleColor="gray",
        titleFontSize=20,
    )
    .properties(width=1000, height=300)
)